# OCR-engine vergelijking voor de modelevaluatie

Om DiffBrush en DiffusionPen te beoordelen lezen we de gegenereerde adresregels terug met een OCR-engine en meten we de CER/WER tegenover de bedoelde tekst. Die OCR-engine moet handschrift betrouwbaar kunnen lezen, anders meten we de fouten van de OCR in plaats van de fouten van het generatieve model.

EasyOCR (de eerste keuze) bleek cursief schrift slecht te lezen: een `K` werd als `y` herkend, een `W` als `L`. Dat geeft een oneerlijk hoge CER. Daarom vergelijken we hier vier OCR-aanpakken en kiezen we de beste.

**De vier aanpakken**
1. **EasyOCR** — detectie + herkenning, getraind op gedrukte tekst / scene text.
2. **TrOCR** (`microsoft/trocr-large-handwritten`) — transformer-OCR, getraind op IAM handschrift. Alleen regel-herkenning, dus we splitsen blokken eerst met horizontale projectie.
3. **TrOCR + EasyOCR-detectie** — EasyOCR vindt de regels, TrOCR herkent ze. De combinatie van beide.
4. **PaddleOCR** — detectie + herkenning, eigen detectie-model.

## Methode

We hebben geen kant-en-klare transcripties van de echte Prime Vision adresblokken. Daarom zijn **12 echte samples uit `data/AddressBlock examples/` handmatig getranscribeerd** (visuele inspectie) en opgeslagen in [`ocr_groundtruth.json`](../data/ocr_groundtruth.json). Dat is de ground truth.

We draaien elke aanpak op die 12 echte handschrift-blokken en berekenen de gemiddelde CER en WER tegenover de transcripties. De engine met de laagste CER op **echt** handschrift gebruiken we vervolgens in `diffbrush_eval.ipynb` en `diffusionpen_eval.ipynb`.

Echt handschrift is hier de juiste testset: het is de distributie die Prime Vision uiteindelijk wil herkennen, en de transcripties zijn zeker.

In [1]:
import json, re
from pathlib import Path

import numpy as np
from PIL import Image
import torch

# Vind de project root (map met data/ocr_groundtruth.json), of de notebook nu
# geopend is vanuit notebooks/ of de project root.
for ROOT in [Path.cwd(), *Path.cwd().parents]:
    if (ROOT / 'data' / 'ocr_groundtruth.json').exists():
        break
else:
    raise FileNotFoundError(
        'data/ocr_groundtruth.json niet gevonden. '
        'Open de notebook vanuit de project root of de notebooks/ map.'
    )

REAL_DIR = ROOT / 'data' / 'AddressBlock examples'
GT = json.loads((ROOT / 'data' / 'ocr_groundtruth.json').read_text(encoding='utf-8'))['samples']
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| ground-truth samples:', len(GT))


# --- metrics ---
def levenshtein(a, b):
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(cur[j - 1] + 1, prev[j] + 1, prev[j - 1] + (ca != cb))
        prev = cur
    return prev[-1]


def norm(s):
    return re.sub(r'\s+', ' ', s).strip().lower()


def cer(ref, hyp):
    ref, hyp = norm(ref), norm(hyp)
    return levenshtein(ref, hyp) / max(1, len(ref))


def wer(ref, hyp):
    rs, hs = norm(ref).split(), norm(hyp).split()
    if not rs:
        return 1.0 if hs else 0.0
    prev = list(range(len(hs) + 1))
    for i, ra in enumerate(rs, 1):
        cur = [i] + [0] * len(hs)
        for j, hb in enumerate(hs, 1):
            cur[j] = min(cur[j - 1] + 1, prev[j] + 1, prev[j - 1] + (ra != hb))
        prev = cur
    return prev[-1] / len(rs)


# --- regel-splitter (TrOCR doet geen detectie) ---
def split_lines(pil):
    """Knip een blok in regels via horizontale projectie van inkt-pixels."""
    g = np.array(pil.convert('L'))
    dark = (g < 160).sum(axis=1)
    if dark.max() == 0:
        return [pil]
    rowmask = dark > max(1.0, dark.max() * 0.06)
    runs, inrun, start = [], False, 0
    for i, v in enumerate(rowmask):
        if v and not inrun:
            inrun, start = True, i
        elif not v and inrun:
            inrun = False
            if i - start > 6:
                runs.append((start, i))
    if inrun and len(rowmask) - start > 6:
        runs.append((start, len(rowmask)))
    crops = []
    for a, b in runs:
        a, b = max(0, a - 6), min(len(rowmask), b + 6)
        crops.append(pil.crop((0, a, pil.width, b)))
    return crops or [pil]

device: cuda:0 | ground-truth samples: 12


In [2]:
# Laad de vier OCR-aanpakken. Eerste run downloadt de modellen.
engines = {}

# 1. EasyOCR
import easyocr
_easy = easyocr.Reader(['en', 'nl'], gpu=torch.cuda.is_available())

def run_easyocr(path):
    return ' '.join(_easy.readtext(str(path), detail=0, paragraph=True)).strip()

engines['EasyOCR'] = run_easyocr
print('EasyOCR geladen')

# 2. TrOCR
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
# transformers 4.57 blokkeert torch.load van .bin checkpoints bij torch < 2.6.
# TrOCR-handwritten heeft geen safetensors-variant; het is een vertrouwde
# officiele Microsoft-release, dus we omzeilen die check bewust.
import transformers.modeling_utils as _mu
_mu.check_torch_load_is_safe = lambda *a, **k: None

_TROCR = 'microsoft/trocr-large-handwritten'
_proc = TrOCRProcessor.from_pretrained(_TROCR)
_trocr = VisionEncoderDecoderModel.from_pretrained(_TROCR).to(DEVICE).eval()

@torch.no_grad()
def trocr_line(pil):
    pv = _proc(pil.convert('RGB'), return_tensors='pt').pixel_values.to(DEVICE)
    ids = _trocr.generate(pv, max_new_tokens=64)
    return _proc.batch_decode(ids, skip_special_tokens=True)[0].strip()

def run_trocr(path):
    crops = split_lines(Image.open(path))
    return ' '.join(trocr_line(c) for c in crops).strip()

engines['TrOCR (projectie-split)'] = run_trocr
print('TrOCR geladen')

# 3. Combo: EasyOCR detecteert regels, TrOCR herkent ze
def run_combo(path):
    pil = Image.open(path).convert('RGB')
    det = _easy.readtext(np.array(pil), detail=1, paragraph=True)
    boxes = []
    for item in det:
        bbox = item[0]
        xs = [p[0] for p in bbox]
        ys = [p[1] for p in bbox]
        boxes.append((min(ys), min(xs), max(ys), max(xs)))
    boxes.sort()
    if not boxes:
        return trocr_line(pil)
    parts = []
    for y0, x0, y1, x1 in boxes:
        crop = pil.crop((max(0, x0 - 4), max(0, y0 - 4), x1 + 4, y1 + 4))
        if crop.width > 6 and crop.height > 6:
            parts.append(trocr_line(crop))
    return ' '.join(parts).strip()

engines['TrOCR + EasyOCR-detectie'] = run_combo
print('Combo geladen')

# 4. PaddleOCR
from paddleocr import PaddleOCR
_paddle = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)

def run_paddle(path):
    res = _paddle.ocr(str(path), cls=True)
    if not res or not res[0]:
        return ''
    items = []
    for box, (text, conf) in res[0]:
        items.append((min(p[1] for p in box), min(p[0] for p in box), text))
    items.sort()
    return ' '.join(t for _, _, t in items).strip()

engines['PaddleOCR'] = run_paddle
print('PaddleOCR geladen')

EasyOCR geladen


C:\Users\karol\OneDrive\Bureaublad\HHS\Jaar 3 Semester 6\Datalab\Prime_Vision\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-large-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TrOCR geladen
Combo geladen


PaddleOCR geladen


In [3]:
# Draai elke aanpak op de 12 echte samples en bereken CER/WER.
try:
    from IPython.display import display
except ImportError:
    display = print

summary, detail = {}, {}
for name, fn in engines.items():
    cers, wers, rows = [], [], []
    for s in GT:
        path = REAL_DIR / s['file']
        try:
            hyp = fn(path)
        except Exception as e:
            hyp = f'<ERROR: {e}>'
        c, w = cer(s['text'], hyp), wer(s['text'], hyp)
        cers.append(c)
        wers.append(w)
        rows.append({'bestand': s['file'], 'referentie': s['text'],
                     'ocr': hyp, 'CER': round(c, 3), 'WER': round(w, 3)})
    summary[name] = {'Gem. CER': round(float(np.mean(cers)), 3),
                     'Gem. WER': round(float(np.mean(wers)), 3)}
    detail[name] = rows
    print(f'{name:30s}  CER {summary[name]["Gem. CER"]:.3f}   WER {summary[name]["Gem. WER"]:.3f}')

best = min(summary, key=lambda k: summary[k]['Gem. CER'])
print(f'\nBeste engine (laagste CER op echt handschrift): {best}')

try:
    import pandas as pd
    tbl = pd.DataFrame(summary).T.sort_values('Gem. CER')
    print('\nSamenvatting (lager = beter):')
    display(tbl)
except ImportError:
    pass

# Sla resultaten op voor het verslag.
out = ROOT / 'results' / 'ocr_benchmark_results.json'
out.write_text(json.dumps({'summary': summary, 'best': best, 'detail': detail},
                          ensure_ascii=False, indent=2), encoding='utf-8')
print('Opgeslagen naar', out)

EasyOCR                         CER 0.438   WER 1.013


TrOCR (projectie-split)         CER 1.214   WER 1.407


TrOCR + EasyOCR-detectie        CER 0.705   WER 1.053


PaddleOCR                       CER 0.349   WER 0.783

Beste engine (laagste CER op echt handschrift): PaddleOCR



Samenvatting (lager = beter):


,Gem. CER,Gem. WER
PaddleOCR,0.349,0.783
EasyOCR,0.438,1.013
TrOCR + EasyOCR-detectie,0.705,1.053
TrOCR (projectie-split),1.214,1.407


Opgeslagen naar C:\Users\karol\OneDrive\Bureaublad\HHS\Jaar 3 Semester 6\Datalab\Prime_Vision\ocr_benchmark_results.json


In [4]:
# Bekijk per sample wat de beste en de slechtste engine eraf lazen.
try:
    import pandas as pd
    worst = max(summary, key=lambda k: summary[k]['Gem. CER'])
    print(f'Beste: {best}   |   slechtste: {worst}\n')
    rows = []
    for b, w in zip(detail[best], detail[worst]):
        rows.append({'referentie': b['referentie'],
                     f'{best}': b['ocr'], f'{worst}': w['ocr']})
    display(pd.DataFrame(rows))
except ImportError:
    for r in detail[best]:
        print(r['referentie'], '->', r['ocr'])

Beste: PaddleOCR   |   slechtste: TrOCR (projectie-split)



,referentie,PaddleOCR,TrOCR (projectie-split)
0,Zuiderzeestraatweg 560 8091 CV Wezep,Zuiderzeestraatweg s6o 80gicy Wezep,not # suggesting see
1,Galileistate 21 2041 BS Zandvoort,galileistate 2i,References and # Gakbeistate v. 1961-000 2041 ...
2,"12410 Pecos Bluff Ct. HUMBLE, TX, 77346 Vereni...",12410 PecosBluffct tluMBETX37346 Verenigde Staten,1961 62 .
3,postbus 30020 2500 GA Den Haag,potbus 30020 2506gA DerHloeg,1961 1957 Pesothu-venillaag 1961 . K. J.
4,Oude Rijksweg 47 4339 BA Nieuw en St. Joosland,Righoweg 47 Oude BA Nieuw enst.Joslomd 4339,"George Byloug""Ten . "" Joostond"
5,Faas Wilkesstraat 2 1095 MD Amsterdam,109s MD Qmste dam,20 Kansas Wieheart at 2
6,Sterbastion 10 1991 PM Velserbroek,SrerbasHoy 10 1991Pm Velrebroeh,1961 Pittsburgh
7,Zwaan 20 3641 WE Mijdrecht,Zxan 20 36u1 WE_mipreclt,References and representatives Ewan 20 0 3641 ...
8,Meester van Lingenlaan 2 App// 152 1942 BJ Bev...,ANE Meester van lingenlacn 2 pp//152 1942Bj Mm...,1961 German banks
9,Koolmees 28 1616 HG Hoogkarspel,Koolmees28 6i6 HG Hoogkarspel,Kortheens Hanskorspd .


## Conclusie

Uit de benchmark op 12 echte Prime Vision adresblokken:

| Engine | Gem. CER | Gem. WER |
|---|---|---|
| **PaddleOCR** | **0.349** | **0.783** |
| EasyOCR | 0.438 | 1.013 |
| TrOCR + EasyOCR-detectie | 0.705 | 1.053 |
| TrOCR (projectie-split) | 1.214 | 1.407 |

**PaddleOCR wint duidelijk** en gebruiken we als OCR-engine in beide eval-notebooks.

**Waarom TrOCR tegenviel** — TrOCR-handwritten is getraind op schone, losse IAM-regels. Op de rommelige echte enveloppen (achtergrondruis, weggelakte vlakken, scheve regels) hallucineert het: het verzint jaartallen als "1961" en "1948". De regel-splitsing op echte blokken is bovendien lastig, wat de combo ook omlaag trok. Op schoon, los gegenereerd handschrift zou TrOCR beter kunnen presteren, maar deze test gaat over de echte doel-distributie.

**Waarom PaddleOCR** — het heeft een eigen detectie-model dat meerdere regels per blok aankan, en de herkenning blijft het dichtst bij de echte tekst ("Zuiderzeestraatweg s6o 80gicy Wezep"). Geen aparte regel-splitsing nodig.

**Kanttekening voor het verslag** — ook PaddleOCR is niet perfect (CER 0.349 op echt handschrift). Een hoge CER op gegenereerde samples kan dus deels de OCR zijn, niet alleen het generatieve model. Visuele inspectie blijft daarom een noodzakelijke tweede metric naast de OCR-readback.